<a href="https://colab.research.google.com/github/Ishan-5/LLM-inference-and-quantization-benchmark/blob/main/LLM_inference_v2ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf /content/llmbench
!git clone https://github.com/Ishan-5/LLM-inference-and-quantization-benchmark.git /content/llmbench
%cd /content/llmbench
!ls configs/benchmark_v2.yaml scripts/run_all_v2.sh COLAB_V2_STEPS.md

Cloning into '/content/llmbench'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (230/230), done.
remote: Total 508 (delta 246), reused 474 (delta 217), pack-reused 0 (from 0)
Receiving objects: 100% (508/508), 306.53 KiB | 4.03 MiB/s, done.
Resolving deltas: 100% (246/246), done.
/content/llmbench
COLAB_V2_STEPS.md  configs/benchmark_v2.yaml  scripts/run_all_v2.sh


In [2]:
!nvidia-smi
!bash scripts/bootstrap_colab.sh

Fri Nov 27 21:22:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
os.environ["PYTHONPATH"] = os.path.abspath("src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python -m llmbench.cli doctor

/usr/local/lib/python3.13/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
llmbench 0.1.0
----------------------------

In [4]:
!python -m llmbench.cli prepare --config configs/benchmark_v2.yaml --backends hf,vllm,awq,vllm_awq

[hf] preparing Qwen/Qwen2.5-1.5B-Instruct
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/1.67M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/8.71M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 242/8.71M [00:00<1:52:40, 1.29kB/s]
Reconstructing (incomplete total...):   0% 242/8.71M [00:00<1:52:40, 1.29kB/s]
Reconstructing (incomplete total...):   0% 242/8.71M [00:00<1:52:40, 1.29kB/s]
Reconstructing (incomplete total...):  15% 1.68M/11.5M [00:00<2:06:52, 1.29kB/s]

Fetching 7 files:  14% 1/7 [00:00<00:01,  4.67it/s]
Reconstructing (incomplete total...):   0% 4.46M/3.10G [00:00<667:06:58, 1.29kB/s]
Reconstructing (incomplete total...):   1% 20.3M/3.10G [00:03<09:46, 5.25MB/s, 47.1MB/s  ]
Re

In [5]:
!python -m llmbench.cli all --config configs/benchmark_v2.yaml --backends hf,vllm,awq,vllm_awq --limit-prompts 8 --results-dir results_v2_smoke


>>> /usr/bin/python3 -m llmbench.cli _worker --kind both --backend hf --config configs/benchmark_v2.yaml --results-dir results_v2_smoke --limit-prompts 8

/usr/local/lib/python3.13/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: Futur

In [6]:
import json, glob, os

for p in sorted(glob.glob("results_v2_smoke/raw/benchmark_*.json")):
    rows = json.load(open(p, encoding="utf-8"))
    print(
        os.path.basename(p),
        "errors=", sum(r.get("errors", 0) for r in rows),
        "out_tokens=", sum(r.get("output_tokens", 0) for r in rows),
        "gpu_ok=", all(r.get("peak_gpu_mib") is not None for r in rows),
        "rows=", len(rows),
    )

for p in sorted(glob.glob("results_v2_smoke/raw/quality_*.json")):
    q = json.load(open(p, encoding="utf-8"))
    print(os.path.basename(p), "MMLU n=", q.get("mmlu_total"), "note=", q.get("mmlu_note") or "ok")

benchmark_awq.json errors= 0 out_tokens= 12288 gpu_ok= True rows= 12
benchmark_hf.json errors= 0 out_tokens= 12288 gpu_ok= True rows= 12
benchmark_vllm.json errors= 0 out_tokens= 12288 gpu_ok= True rows= 12
benchmark_vllm_awq.json errors= 0 out_tokens= 12288 gpu_ok= True rows= 12
quality_awq.json MMLU n= 200 note= ok
quality_hf.json MMLU n= 200 note= ok
quality_vllm.json MMLU n= 200 note= ok
quality_vllm_awq.json MMLU n= 200 note= ok


In [7]:
!BACKENDS=hf,vllm,awq,vllm_awq bash scripts/run_all_v2.sh

/usr/local/lib/python3.13/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
llmbench 0.1.0
----------------------------

In [8]:
import json, glob, os

for p in sorted(glob.glob("results_v2/raw/benchmark_*.json")):
    rows = json.load(open(p, encoding="utf-8"))
    print(
        os.path.basename(p),
        "errors=", sum(r.get("errors", 0) for r in rows),
        "out_tokens=", sum(r.get("output_tokens", 0) for r in rows),
        "rows=", len(rows),
    )

for p in sorted(glob.glob("results_v2/raw/quality_*.json")):
    q = json.load(open(p, encoding="utf-8"))
    print(os.path.basename(p), "MMLU n=", q.get("mmlu_total"), "note=", q.get("mmlu_note") or "ok")

benchmark_awq.json errors= 0 out_tokens= 49152 rows= 12
benchmark_hf.json errors= 0 out_tokens= 48985 rows= 12
benchmark_vllm.json errors= 0 out_tokens= 48986 rows= 12
benchmark_vllm_awq.json errors= 0 out_tokens= 49152 rows= 12
quality_awq.json MMLU n= 200 note= ok
quality_hf.json MMLU n= 200 note= ok
quality_vllm.json MMLU n= 200 note= ok
quality_vllm_awq.json MMLU n= 200 note= ok


In [9]:
!zip -rq results_v2.zip results_v2
from google.colab import files
files.download("results_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>